# Songo Lot 38 — Colab compute benchmark
Ce notebook orchestre uniquement le script versionné. Aucun entraînement n'est exécuté.

In [6]:
import os, pathlib, subprocess, sys
print('COLAB:', 'COLAB_RELEASE_TAG' in os.environ)
print('Python:', sys.version)
subprocess.run(['nvidia-smi'], check=False)

COLAB: True
Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


CompletedProcess(args=['nvidia-smi'], returncode=0)

## Récupération et installation du dépôt
Cette cellule clone le dépôt dans `/content/songo-fish`, puis place Colab à la racine contenant `pyproject.toml`. Les checkpoints G4 doivent ensuite être copiés séparément sous `data/experiments/lot34r_g4_retry/checkpoints/`.

In [7]:
from pathlib import Path
import os, subprocess, sys
REPO_URL = 'https://github.com/GlennEriss/songo-fish.git'
REPO_DIR = Path('/content/songo-fish')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
os.chdir(REPO_DIR)
assert Path('pyproject.toml').is_file(), f'pyproject.toml absent de {Path.cwd()}'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[train,perf,dev]'], check=True)
os.environ['PYTHONPATH'] = 'packages:apps/trainer/scripts'
print('Projet installé depuis :', Path.cwd())

Projet installé depuis : /content/songo-fish


In [ ]:
# Google Drive partagé avec le bridge local macOS
from google.colab import drive
drive.mount('/content/drive')
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/songo-ai')
INPUT_BUNDLE = DRIVE_PROJECT_ROOT / 'inputs/lot38_inputs.tar.gz'
OUTPUT_DIR = DRIVE_PROJECT_ROOT / 'experiments/lot38_colab_compute'
EXPORT_BUNDLE = DRIVE_PROJECT_ROOT / 'exports/lot38_results.tar.gz'
assert INPUT_BUNDLE.is_file(), f'Bundle Drive absent : {INPUT_BUNDLE}'
bundle = INPUT_BUNDLE
subprocess.run(['tar', '-xzf', str(bundle), '-C', str(REPO_DIR)], check=True)
assert Path('data/experiments/lot35_generator_pool/g4_champion_identity.json').is_file()
assert Path('data/experiments/lot34r_g4_retry/checkpoints/pool/step-06000.pt').is_file()
assert Path('data/experiments/lot34r_g4_retry/checkpoints/pool/step-12000.pt').is_file()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_BUNDLE.parent.mkdir(parents=True, exist_ok=True)
print('Entrées POOL_G4R installées ; sorties persistantes :', OUTPUT_DIR)

In [ ]:
!python apps/trainer/scripts/run_srn_colab_benchmark.py --stage prepare --device auto --output "$OUTPUT_DIR"

In [ ]:
# Smoke test optionnel : remplacer 4096 par 1024
!python apps/trainer/scripts/run_srn_colab_benchmark.py --stage benchmark --device auto --budget 4096 --output "$OUTPUT_DIR"

In [ ]:
!python apps/trainer/scripts/run_srn_colab_benchmark.py --stage finalize --output "$OUTPUT_DIR"
!python apps/trainer/scripts/run_srn_colab_benchmark.py --stage export --output "$OUTPUT_DIR" --bundle "$EXPORT_BUNDLE"

Le bundle et son fichier `.sha256` sont écrits dans `songo-ai/exports/` sur Drive. Le Mac peut ensuite les importer avec `python apps/trainer/scripts/import_remote_experiment.py --experiment lot38 --from-drive --wait-for-drive --timeout 3600`.